# Evolving sorting networks through coevolution

In this lab session we are going to leverage **coevolution** to evolve **sorting networks**.

Sorting networks are algorithms designed to arrange a sequence of elements into a specific order, usually ascending or descending. The key feature of sorting networks is that they use a fixed set of comparisons and swaps to achieve the sorting, and this set of operations is independent of the input data ([wikipedia](https://en.wikipedia.org/wiki/Sorting_network)).

We will represent networks as lists of comparators, where each comparator is a pair of indices indicating which elements to compare.

First of all, we import the random module and set the seed.

In [153]:
import random
random.seed(0)

Let us define a function which sorts an array with a given sorting network.

In [154]:
def eval_sorting_network(sn, array):
    array = array.copy()
    for i, j in sn:
        if array[i] > array[j]:
            array[i], array[j] = array[j], array[i]
    return array

In [155]:
sn=[(0,2), (1,3), (0,3), (1,2), (0,1), (2,3)]
eval_sorting_network(sn, [1,7,2,3])

[1, 2, 3, 7]

Now, we write the code to initialize the 2 competing populations: one for the networks and one for the arrays.

In [156]:
def get_random_network(value_range, depth):
    return [tuple(sorted(random.sample(value_range, k=2))) for _ in range(depth)]

In [157]:
def init_array_population(value_range, dim, pop_size):
    return [random.choices(value_range, k=dim) for _ in range(0, pop_size)]

def init_network_population(value_range, min_depth, max_depth, pop_size):
    pop = []
    for _ in range(pop_size):
        depth = random.choice(range(min_depth, max_depth+1))
        pop.append(get_random_network(value_range, depth))
    return pop

Let us define the 1 fitness functions to compute all the fitness of each network and of each array. A network will have good fitness if it can sort many arrays, while an array will have good fitness if it can 'trick' many networks.

In [158]:
def overall_fitness(net_pop, arr_pop):
    net_scores = [0]*len(net_pop)
    arr_scores = [0]*len(arr_pop)
    for i, arr in enumerate(arr_pop):
        for j, sn in enumerate(net_pop):
            if eval_sorting_network(sn, arr) == sorted(arr): # if the array is sorted correctly, we increase the score
                net_scores[j] += 1
            else:
                arr_scores[i] += 1
    return net_scores, arr_scores

We now implement the tournament selection and the crossover. We can use the same functions for both the populations.

In [167]:
def tournament_selection(pop, scores, k):
  print(len(pop))
  tournament = random.choices(range(len(pop)), k=k)
  
  selected = max([(scores[idx], pop[idx]) for idx in tournament])
  return selected[1]

In [169]:
def crossover(x, y):
  # random indexes over the possible indexes of the individuals
  psx = sorted(random.choices([i for i in range (1, len(x))], k=2))
  psy = sorted(random.choices([i for i in range (1, len(y))], k=2))
  # offspring 1 
  offspring1 = x[:psx[0]]
  offspring1.extend(y[psy[0]:psy[1]])
  offspring1.extend(x[psx[1]:])
  return offspring1

We can now implement a mutation function for the arrays.

In [161]:
def array_mutation(x, value_range, p_m):
  def mutate(v):
    if random.random() < p_m:
        res = random.choice(value_range)
        while res == v: # we avoid sampling the same value
            res = random.choice(value_range)
        return res
    else:
      return v
  return [mutate(v) for v in x]

Now, we can choose one or more mutation operators for the sorting networks. You can use different ones during the evolution.

In [162]:
def sn_mutation(sn, value_range, p_m): # the arguments may change depending on the specific kind of mutation
    for i, _ in enumerate(sn):
        if random.uniform(0, 1) <= p_m:
            new_start = random.sample(value_range, k=1)
            new_end = random.sample(value_range, k=1)
            t = new_start
            new_start = min(new_start, new_end)
            new_end = max(new_end, t)
            sn[i] = (new_start[0], new_end[0])
    return sn

We have now all the elements to write the code for a generation.

In [163]:
def get_best(pop, scores):
  return max(list(zip(scores, pop)))

In [ ]:
def generation(net_pop, arr_pop, net_scores, arr_scores, crossover, arr_dim, value_range, p_m, t_size):
  pop_size = len(net_pop)
  # perform selection for both the populations
  selected_net = [tournament_selection(net_pop, net_scores, t_size) for _ in range(0, pop_size)]
  selected_arr = [tournament_selection(arr_pop, arr_scores, t_size) for _ in range(0, pop_size)]
  # perform crossover
  pairs_net = zip(selected_net, selected_net[1:] + selected_net[0:1])
  pairs_arr = zip(selected_arr, selected_arr[1:] + selected_arr[0:1])
  offspring_net = [crossover(*pair) for pair in pairs_net]
  
  offspring_arr = []
  for pair in pairs_arr:
    off1, off2 = crossover(*pair)
    offspring_arr.append(off1)
  # apply the mutation operator(s) to the offspring
  net_pop = list(map(lambda x: sn_mutation(x, range(arr_dim), p_m), offspring_net))
  arr_pop = list(map(lambda x: array_mutation(x, value_range, p_m), offspring_arr))
  return net_pop, arr_pop

We can now define our `coevolution` function.

In [165]:
def coevolution(value_range, 
                pop_size,
                arr_dim,
                min_depth,
                max_depth,
                overall_fitness,
                crossover,
                t_size = 10, 
                n_gen = 200):
  
  p_m = 1/arr_dim
  # initialize the population
  Pt = init_network_population(range(arr_dim), min_depth, max_depth, pop_size)
  Qt = init_array_population(value_range, arr_dim, pop_size)

  net_scores, arr_scores = overall_fitness(Pt, Qt)
  net_history = [get_best(Pt, net_scores)[1]]
  arr_history = [get_best(Qt, arr_scores)[1]]
  
  for _ in range(0, n_gen):
    Pt, Qt = generation(Pt, Qt, net_scores, arr_scores, crossover, arr_dim, value_range, p_m, t_size)
    net_history.append(get_best(Pt, net_scores)[1])
    arr_history.append(get_best(Qt, arr_scores)[1])
    
  return get_best(Pt, net_scores)[1], get_best(Qt, arr_scores)[1], net_history, arr_history

Try your code for different array dimensions and parameters.

In [168]:
best_net, best_arr, net_history, arr_history = coevolution(
    value_range=range(50),
    pop_size= 20,
    arr_dim= 10,
    min_depth= 5,
    max_depth= 15,
    overall_fitness=overall_fitness,
    crossover=crossover,
    t_size = 3,
    n_gen = 100
    )

20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
20
40


IndexError: list index out of range

In [ ]:
print(best_net)
print(best_arr)

In [ ]:
random_arr = random.sample(range(50), k = 10)
print(random_arr)
eval_sorting_network(best_net, random_arr)

In [ ]:
eval_sorting_network(best_net, best_arr)

Plot the evolution of the fitness score of the best individual for both populations

In [ ]:
import matplotlib.pyplot as plt

In [ ]:
plt.plot(net_history)
plt.ylabel('Network fitness')
plt.xlabel('Generation')
plt.show()

In [ ]:
plt.plot(arr_history)
plt.ylabel('Array fitness')
plt.xlabel('Generation')
plt.show()